# SICOR — seleção das glebas do Cerrado (2021–2025)

Processa cinco bases anuais:

- `sicor_glebas_wkt_2021.csv`
- `sicor_glebas_wkt_2022.csv`
- `sicor_glebas_wkt_2023.csv`
- `sicor_glebas_wkt_2024.csv`
- `sicor_glebas_wkt_2025.csv`

## O que o notebook faz

1. Monta o Google Drive.
2. Carrega o limite oficial do bioma Cerrado pelo pacote `geobr`.
3. Lê cada CSV em blocos, evitando carregar os cinco arquivos completos simultaneamente.
4. Converte `GT_GEOMETRIA` de WKT para geometria.
5. Mantém somente as glebas espacialmente contidas no Cerrado.
6. Acrescenta a coluna `ANO_BASE`.
7. Gera um CSV por ano e um CSV consolidado de 2021 a 2025.
8. Produz uma tabela-resumo da execução.

Por padrão, o critério espacial é `within`, igual ao notebook original: a gleba precisa estar inteiramente dentro do Cerrado. O parâmetro pode ser alterado para `intersects` para incluir glebas que cruzem o limite do bioma.

In [ ]:
# ============================================================
# 1. INSTALAÇÃO E IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================================

!pip install -q --upgrade geopandas geobr pyogrio shapely

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import gc
import os
import warnings

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import geobr

from shapely import from_wkt

warnings.filterwarnings("ignore", category=UserWarning)

print("Bibliotecas carregadas com sucesso.")
print("GeoPandas:", gpd.__version__)

## Configuração

Os caminhos abaixo correspondem, no Google Colab, ao caminho do Windows:

```text
G:\Meu Drive\Colab Notebooks\project\sicor_preparacao\dados_bcb
```

No Colab, esse caminho aparece como:

```text
/content/drive/MyDrive/Colab Notebooks/project/sicor_preparacao/dados_bcb
```

In [ ]:
# ============================================================
# 2. PARÂMETROS
# ============================================================

PASTA_ENTRADA = Path(
    "/content/drive/MyDrive/Colab Notebooks/project/"
    "sicor_preparacao/dados_bcb"
)

PASTA_SAIDA = Path(
    "/content/drive/MyDrive/Colab Notebooks/project/output/"
    "sicor_cerrado_2021_2025"
)

ANOS = [2021, 2022, 2023, 2024, 2025]

ARQUIVOS_ENTRADA = {
    ano: PASTA_ENTRADA / f"sicor_glebas_wkt_{ano}.csv"
    for ano in ANOS
}

ARQUIVOS_SAIDA_ANUAIS = {
    ano: PASTA_SAIDA / f"sicor_cerrado_{ano}.csv"
    for ano in ANOS
}

ARQUIVO_SAIDA_CONSOLIDADO = (
    PASTA_SAIDA / "sicor_cerrado_2021_2025.csv"
)

ARQUIVO_RESUMO = (
    PASTA_SAIDA / "resumo_processamento_sicor_cerrado_2021_2025.csv"
)

# Quantidade de registros lidos a cada bloco.
# Reduza para 25_000 se o Colab ficar sem memória.
TAMANHO_BLOCO = 100_000

# Mantém o comportamento do notebook original.
# Use "intersects" para incluir glebas que apenas cruzem o Cerrado.
CRITERIO_ESPACIAL = "within"

# O CSV original utiliza ponto e vírgula.
SEPARADOR = ";"

# Codificações testadas automaticamente.
CODIFICACOES = ["utf-8-sig", "utf-8", "latin-1", "cp1252"]

# Coluna que contém a geometria WKT.
COLUNA_WKT = "GT_GEOMETRIA"

PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

print("Pasta de entrada:", PASTA_ENTRADA)
print("Pasta de saída:", PASTA_SAIDA)
print("Critério espacial:", CRITERIO_ESPACIAL)

In [ ]:
# ============================================================
# 3. VERIFICAÇÃO DOS ARQUIVOS DE ENTRADA
# ============================================================

arquivos_ausentes = []

for ano, caminho in ARQUIVOS_ENTRADA.items():
    existe = caminho.exists()
    tamanho_gb = caminho.stat().st_size / (1024**3) if existe else None

    print(
        f"{ano}: {'OK' if existe else 'NÃO ENCONTRADO'}"
        + (f" | {tamanho_gb:.2f} GB" if existe else "")
        + f" | {caminho}"
    )

    if not existe:
        arquivos_ausentes.append(str(caminho))

if arquivos_ausentes:
    raise FileNotFoundError(
        "Os seguintes arquivos não foram encontrados:\n"
        + "\n".join(arquivos_ausentes)
    )

print("\nTodos os cinco arquivos foram encontrados.")

In [ ]:
# ============================================================
# 4. LEITURA DO LIMITE OFICIAL DO BIOMA CERRADO
# ============================================================

biomas = geobr.read_biomes(year=2019)

# Compatibilidade com possíveis variações do nome da coluna.
coluna_nome_bioma = next(
    (
        coluna
        for coluna in ["name_biome", "name_bioma", "biome", "bioma"]
        if coluna in biomas.columns
    ),
    None
)

if coluna_nome_bioma is None:
    raise KeyError(
        "Não foi possível identificar a coluna com o nome do bioma. "
        f"Colunas disponíveis: {list(biomas.columns)}"
    )

cerrado = biomas[
    biomas[coluna_nome_bioma]
    .astype(str)
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("ascii")
    .str.strip()
    .str.lower()
    .eq("cerrado")
].copy()

if cerrado.empty:
    raise ValueError("O bioma Cerrado não foi localizado na base do geobr.")

# Dissolve todas as partes do Cerrado em uma única feição.
# Isso evita duplicação de glebas durante a junção espacial.
cerrado = cerrado.to_crs("EPSG:4326")
cerrado = cerrado[["geometry"]].dissolve().reset_index(drop=True)

print("CRS do Cerrado:", cerrado.crs)
print("Quantidade de feições após dissolve:", len(cerrado))

ax = cerrado.boundary.plot(figsize=(9, 9), linewidth=0.8)
ax.set_title("Limite do bioma Cerrado utilizado no processamento")
ax.set_axis_off()
plt.show()

In [ ]:
# ============================================================
# 5. FUNÇÕES AUXILIARES
# ============================================================

def detectar_codificacao(caminho, separador=";"):
    """Testa as codificações configuradas usando apenas cinco linhas."""
    ultimo_erro = None

    for codificacao in CODIFICACOES:
        try:
            teste = pd.read_csv(
                caminho,
                sep=separador,
                encoding=codificacao,
                nrows=5
            )

            if len(teste.columns) <= 1:
                continue

            return codificacao, list(teste.columns)

        except Exception as erro:
            ultimo_erro = erro

    raise RuntimeError(
        f"Não foi possível ler {caminho}. Último erro: {ultimo_erro}"
    )


def converter_wkt_para_geometria(serie_wkt):
    """Converte WKT preservando exatamente o índice original do bloco."""
    textos = serie_wkt.astype("string").fillna("").to_numpy()

    return gpd.GeoSeries(
        from_wkt(textos, on_invalid="ignore"),
        index=serie_wkt.index,
        crs="EPSG:4326"
    )


def corrigir_geometrias_invalidas(gdf):
    """Tenta corrigir geometrias inválidas sem interromper o processamento."""
    mascara_invalida = (
        gdf.geometry.notna()
        & ~gdf.geometry.is_empty
        & ~gdf.geometry.is_valid
    )

    quantidade = int(mascara_invalida.sum())

    if quantidade:
        try:
            gdf.loc[mascara_invalida, "geometry"] = (
                gdf.loc[mascara_invalida, "geometry"].make_valid()
            )
        except Exception:
            # Alternativa compatível com versões anteriores.
            gdf.loc[mascara_invalida, "geometry"] = (
                gdf.loc[mascara_invalida, "geometry"].buffer(0)
            )

    return gdf, quantidade


def preparar_arquivo_saida(caminho):
    """Remove uma saída antiga para impedir duplicação ao executar novamente."""
    if caminho.exists():
        caminho.unlink()
        print("Saída anterior removida:", caminho.name)


def exportar_bloco_csv(df, caminho, escrever_cabecalho):
    """Acrescenta um bloco ao CSV usando ponto e vírgula e UTF-8-SIG."""
    df.to_csv(
        caminho,
        mode="a",
        index=False,
        sep=SEPARADOR,
        encoding="utf-8-sig",
        header=escrever_cabecalho
    )

In [ ]:
# ============================================================
# 6. PREPARAÇÃO DAS SAÍDAS
# ============================================================

for caminho in ARQUIVOS_SAIDA_ANUAIS.values():
    preparar_arquivo_saida(caminho)

preparar_arquivo_saida(ARQUIVO_SAIDA_CONSOLIDADO)
preparar_arquivo_saida(ARQUIVO_RESUMO)

print("Arquivos de saída preparados.")

In [ ]:
# ============================================================
# 7. PROCESSAMENTO DOS CINCO ANOS
# ============================================================

resumos = []
cabecalho_consolidado = True

for ano in ANOS:
    caminho_entrada = ARQUIVOS_ENTRADA[ano]
    caminho_saida_anual = ARQUIVOS_SAIDA_ANUAIS[ano]

    print("\n" + "=" * 80)
    print(f"PROCESSANDO O ANO {ano}")
    print("=" * 80)

    codificacao, colunas = detectar_codificacao(caminho_entrada, SEPARADOR)

    if COLUNA_WKT not in colunas:
        raise KeyError(
            f"A coluna {COLUNA_WKT!r} não existe em {caminho_entrada.name}. "
            f"Colunas encontradas: {colunas}"
        )

    print("Codificação detectada:", codificacao)
    print("Quantidade de colunas:", len(colunas))

    total_lido = 0
    total_wkt_invalido = 0
    total_geometrias_corrigidas = 0
    total_selecionado = 0
    numero_bloco = 0
    cabecalho_anual = True

    leitor = pd.read_csv(
        caminho_entrada,
        sep=SEPARADOR,
        encoding=codificacao,
        chunksize=TAMANHO_BLOCO,
        low_memory=False
    )

    for bloco in leitor:
        numero_bloco += 1
        total_lido += len(bloco)

        # Mantém o texto WKT original no CSV final.
        geometria = converter_wkt_para_geometria(bloco[COLUNA_WKT])

        total_wkt_invalido += int(
            geometria.isna().sum() + geometria.is_empty.fillna(False).sum()
        )

        # A coluna espacial é temporária e não substitui GT_GEOMETRIA.
        gdf_bloco = gpd.GeoDataFrame(
            bloco,
            geometry=geometria,
            crs="EPSG:4326"
        )

        gdf_bloco = gdf_bloco[
            gdf_bloco.geometry.notna()
            & ~gdf_bloco.geometry.is_empty
        ].copy()

        gdf_bloco, corrigidas = corrigir_geometrias_invalidas(gdf_bloco)
        total_geometrias_corrigidas += corrigidas

        # Filtro preliminar pela caixa envolvente do Cerrado.
        minx, miny, maxx, maxy = cerrado.total_bounds
        limites = gdf_bloco.geometry.bounds

        mascara_bbox = (
            (limites["maxx"] >= minx)
            & (limites["minx"] <= maxx)
            & (limites["maxy"] >= miny)
            & (limites["miny"] <= maxy)
        )

        gdf_candidatas = gdf_bloco.loc[mascara_bbox].copy()

        if gdf_candidatas.empty:
            print(
                f"Bloco {numero_bloco:03d}: "
                f"lidas={len(bloco):,} | candidatas=0 | Cerrado=0"
            )
            del bloco, geometria, gdf_bloco, gdf_candidatas
            gc.collect()
            continue

        # Junção espacial com a feição única do Cerrado.
        selecionadas = gpd.sjoin(
            gdf_candidatas,
            cerrado,
            how="inner",
            predicate=CRITERIO_ESPACIAL
        )

        # Remove colunas temporárias da operação espacial.
        selecionadas = selecionadas.drop(
            columns=["geometry", "index_right"],
            errors="ignore"
        )

        # Identifica explicitamente o ano de origem.
        selecionadas["ANO_BASE"] = ano

        quantidade_selecionada = len(selecionadas)
        total_selecionado += quantidade_selecionada

        if quantidade_selecionada:
            exportar_bloco_csv(
                selecionadas,
                caminho_saida_anual,
                escrever_cabecalho=cabecalho_anual
            )

            exportar_bloco_csv(
                selecionadas,
                ARQUIVO_SAIDA_CONSOLIDADO,
                escrever_cabecalho=cabecalho_consolidado
            )

            cabecalho_anual = False
            cabecalho_consolidado = False

        print(
            f"Bloco {numero_bloco:03d}: "
            f"lidas={len(bloco):,} | "
            f"candidatas={len(gdf_candidatas):,} | "
            f"Cerrado={quantidade_selecionada:,}"
        )

        del (
            bloco,
            geometria,
            gdf_bloco,
            gdf_candidatas,
            selecionadas
        )
        gc.collect()

    percentual = (
        100 * total_selecionado / total_lido
        if total_lido
        else 0
    )

    resumo_ano = {
        "ANO_BASE": ano,
        "ARQUIVO_ENTRADA": caminho_entrada.name,
        "CODIFICACAO": codificacao,
        "TOTAL_LIDO": total_lido,
        "WKT_INVALIDO_OU_VAZIO": total_wkt_invalido,
        "GEOMETRIAS_CORRIGIDAS": total_geometrias_corrigidas,
        "TOTAL_CERRADO": total_selecionado,
        "PERCENTUAL_CERRADO": percentual,
        "CRITERIO_ESPACIAL": CRITERIO_ESPACIAL,
        "ARQUIVO_SAIDA": caminho_saida_anual.name
    }

    resumos.append(resumo_ano)

    print(f"\nResumo de {ano}:")
    print(f"Registros lidos: {total_lido:,}")
    print(f"Glebas selecionadas: {total_selecionado:,}")
    print(f"Percentual no Cerrado: {percentual:.2f}%")
    print(f"Saída: {caminho_saida_anual}")

print("\nProcessamento dos cinco anos concluído.")

In [ ]:
# ============================================================
# 7.1. VALIDAÇÃO DO PROCESSAMENTO COMPLETO
# ============================================================

# A base de 2025 isoladamente já possuía aproximadamente 98 mil glebas
# selecionadas no Cerrado em processamento anterior. Por isso, um total
# consolidado de cinco anos inferior a esse valor deve ser investigado.

total_consolidado = sum(item["TOTAL_CERRADO"] for item in resumos)

print(f"Total consolidado de glebas selecionadas: {total_consolidado:,}")

if total_consolidado < 98_000:
    print(
        "\nATENÇÃO: o total consolidado ficou abaixo de 98 mil registros. "
        "Verifique os arquivos de entrada, o critério espacial, mensagens de "
        "erro e o resumo por ano antes de utilizar o resultado."
    )
else:
    print(
        "\nValidação preliminar aprovada: o total consolidado é compatível "
        "com o fato de que 2025, isoladamente, já possuía cerca de 98 mil "
        "glebas no Cerrado."
    )

In [ ]:
# ============================================================
# 8. RESUMO CONSOLIDADO
# ============================================================

df_resumo = pd.DataFrame(resumos)

linha_total = {
    "ANO_BASE": "TOTAL",
    "ARQUIVO_ENTRADA": "2021 a 2025",
    "CODIFICACAO": "-",
    "TOTAL_LIDO": df_resumo["TOTAL_LIDO"].sum(),
    "WKT_INVALIDO_OU_VAZIO": df_resumo["WKT_INVALIDO_OU_VAZIO"].sum(),
    "GEOMETRIAS_CORRIGIDAS": df_resumo["GEOMETRIAS_CORRIGIDAS"].sum(),
    "TOTAL_CERRADO": df_resumo["TOTAL_CERRADO"].sum(),
    "PERCENTUAL_CERRADO": (
        100 * df_resumo["TOTAL_CERRADO"].sum()
        / df_resumo["TOTAL_LIDO"].sum()
        if df_resumo["TOTAL_LIDO"].sum()
        else 0
    ),
    "CRITERIO_ESPACIAL": CRITERIO_ESPACIAL,
    "ARQUIVO_SAIDA": ARQUIVO_SAIDA_CONSOLIDADO.name
}

df_resumo_com_total = pd.concat(
    [df_resumo, pd.DataFrame([linha_total])],
    ignore_index=True
)

df_resumo_com_total.to_csv(
    ARQUIVO_RESUMO,
    sep=SEPARADOR,
    index=False,
    encoding="utf-8-sig"
)

display(df_resumo_com_total)

print("\nArquivo consolidado:")
print(ARQUIVO_SAIDA_CONSOLIDADO)

print("\nResumo do processamento:")
print(ARQUIVO_RESUMO)

In [ ]:
# ============================================================
# 9. VALIDAÇÃO DOS ARQUIVOS GERADOS
# ============================================================

arquivos_gerados = [
    *ARQUIVOS_SAIDA_ANUAIS.values(),
    ARQUIVO_SAIDA_CONSOLIDADO,
    ARQUIVO_RESUMO
]

for caminho in arquivos_gerados:
    if caminho.exists():
        tamanho_mb = caminho.stat().st_size / (1024**2)
        print(f"OK | {caminho.name} | {tamanho_mb:,.2f} MB")
    else:
        print(f"NÃO GERADO | {caminho}")

# Lê apenas cinco linhas do consolidado para conferência.
amostra = pd.read_csv(
    ARQUIVO_SAIDA_CONSOLIDADO,
    sep=SEPARADOR,
    encoding="utf-8-sig",
    nrows=5
)

print("\nColunas do arquivo consolidado:")
print(list(amostra.columns))

display(amostra)

## Saídas esperadas

O notebook cria a pasta:

```text
G:\Meu Drive\Colab Notebooks\project\output\sicor_cerrado_2021_2025
```

e grava:

```text
sicor_cerrado_2021.csv
sicor_cerrado_2022.csv
sicor_cerrado_2023.csv
sicor_cerrado_2024.csv
sicor_cerrado_2025.csv
sicor_cerrado_2021_2025.csv
resumo_processamento_sicor_cerrado_2021_2025.csv
```

O consolidado mantém todas as colunas originais e acrescenta `ANO_BASE`.

### Critério espacial

- `within`: mantém somente glebas inteiramente contidas no Cerrado, reproduzindo o comportamento do notebook antigo.
- `intersects`: inclui também glebas parcialmente sobrepostas ao Cerrado.

Para mudar o critério, altere apenas:

```python
CRITERIO_ESPACIAL = "intersects"
```